# Análisis Exploratorio de Datos (EDA) — Taller 2

## Caso: Focos de dengue en el Valle del Cauca (2025)

**Grupo 3**

| Integrante |
|------------|
| Juan Manuel Román Villa |
| Dora Valencia Martínez |
| Julian Aguilar Mayorga |
| Camilo Percy Ocampo |
| Viviana Fernández Payan |
| Giovanni Jaramillo Bolaños |
| Victor Manuel Hurtado López |

---

Este cuaderno es el **entregable completo** del Taller 2: problema, justificación, pregunta SMART e EDA. Sigue la misma estructura metodológica del Taller 1 y de la guía del profesor, **bajando el alcance** de lo nacional a **un año y una población territorial**.

> **Pregunta SMART + Hipótesis → Arquitectura de datos → Diccionario → Visión general → Limpieza → Análisis univariado → Análisis bivariado → Conclusiones**

**Qué cambia respecto al Taller 1**

| Taller 1 | Taller 2 (este notebook) |
|----------|--------------------------|
| País, año 2025 | **Valle del Cauca**, año **2025** |
| Ranking de departamentos | Ranking de **municipios** (focos epidemiológicos) |
| H3 = Valle vs nacional | H3 = perfil **dentro del Valle** (edad, sexo, hospitalización, gestantes) |
| SMART de proyecto (precisión 80 %, mortalidad) | SMART **medible con este EDA** (concentración municipal, ventana temporal, carga clínica) |

**Fuente única:** Excel SIVIGILA `Datos_2025_210.xlsx` (evento 210 — Dengue). El universo de análisis, tras la limpieza, son las notificaciones con **ocurrencia en el Valle del Cauca** y `FEC_NOT` en 2025.

**Arquitectura (rúbrica Taller 2):** sección 2 de este notebook y PDF [`docs/Propuesta_arquitectura_datos_dengue_taller2.pdf`](../docs/Propuesta_arquitectura_datos_dengue_taller2.pdf).


---
# 1. Definición del problema, justificación y Pregunta SMART

> *Etapa del flujo: **Pregunta SMART + Hipótesis***

## 1.1. Tabla de análisis del problema

| Campo | Contenido |
|-------|-----------|
| **Descripción del problema e impacto (con métrica)** | En el Valle del Cauca el dengue genera **sobrecarga hospitalaria municipal** (Cali y otros focos). El Taller 1 mostró que Valle ya tiene un perfil distinto al nacional (~7.4k casos; hosp. menor; edad más alta). Aquí **no** repetimos el ranking nacional: caracterizamos **dónde, cuándo y en quién** se concentran los casos **dentro del departamento**. **KPI medibles (Excel 2025, Valle):** volumen de casos; % confirmados; % hospitalización; share del top 10 municipios; ventana de pico (mes/semana); perfil edad/sexo; casos en **gestantes**. **No medimos mortalidad:** `FEC_DEF` está 100 % nulo. |
| **Tipo de analítica** | **Descriptiva / diagnóstica** en este taller (priorizar municipios y poblaciones). La visión del proyecto sigue siendo **predictiva** (regresión municipio–semana), pero el EDA se acota al Valle 2025. |
| **Caso similar (estado del arte)** | Delpino et al. (2026): severidad, hospitalización y mortalidad. Martin et al. (2026): clima como predictor (visión futura). Kumar et al. (2026): ML para alertas tempranas. |
| **Tipo de problema de IA** | **Regresión** a futuro: casos agregados por **municipio–semana en el Valle**. Este notebook no entrena el modelo. |
| **Datos usados en este taller** | **Única fuente:** `Datos_2025_210.xlsx`. Se carga el archivo nacional y se **filtra** a ocurrencia en Valle del Cauca y `FEC_NOT` 2025. |
| **Impacto en el negocio con métricas** | 1) Priorizar municipios-foco (concentración del top 10). 2) Anticipar la ventana de pico intra-anual. 3) Diferenciar carga clínica y poblaciones (niños, adultos, gestantes) para la red de atención. |
| **Pregunta SMART** | Ver sección 1.3. |

## 1.2. Complejidad del problema

| Aspecto | Descripción |
|---------|-------------|
| **Variables involucradas** | Tiempo (`SEMANA`, `FEC_NOT`, `mes`); municipio de ocurrencia; edad/sexo; hospitalización; confirmación; `GP_GESTAN` / `sem_ges`; `AREA` |
| **Procesos involucrados** | Vigilancia epidemiológica departamental, hospitalización municipal, priorización de focos y de poblaciones especiales |
| **Dificultad técnica** | Media: fechas en texto, `UNI_MED` mixto, nulos estructurales, filtro territorial sobre nombres/códigos, `sem_ges` vacío en no gestantes |

## 1.3. Pregunta SMART

> **Pregunta SMART:** En el **Valle del Cauca** durante el año epidemiológico **2025**, ¿un conjunto reducido de municipios (top 10) concentra al menos el **60 %** de las notificaciones de dengue, y esa carga se agrupa en una **ventana de pico identificable** (meses/semanas epidemiológicas), con un perfil demográfico y clínico (edad, sexo, **% hospitalización**, población **gestante**) que permita a la Secretaría de Salud **priorizar focos municipales** en ese mismo año?

### Desglose SMART

| Criterio | Cómo se cumple |
|----------|----------------|
| **Específica** | Dengue, Valle del Cauca, municipios-foco y poblaciones (edad/sexo/gestantes), fuente SIVIGILA 2025. |
| **Medible** | % de casos en el top 10 municipal; mes/semana de máximo; % hospitalización; edad media/mediana; % por sexo; n y % de gestantes. |
| **Alcanzable** | Todo se calcula con el Excel 2025 tras filtrar ocurrencia en Valle. No exige modelo ni otras fuentes. |
| **Relevante** | Orienta vigilancia y capacidad hospitalaria **donde ocurren** los casos, no el ranking nacional del Taller 1. |
| **Temporal** | Acotada al año **2025** (un ciclo epidemiológico del archivo). |

### Qué responde este EDA vs qué queda para modelado

| Responde el notebook ahora | Queda para fases posteriores |
|----------------------------|------------------------------|
| Volumen y calidad de casos **en Valle 2025** | Precisión de un modelo > 80 % |
| Estacionalidad intra-anual en el departamento (H1) | IDEAM / DANE u otras fuentes |
| Concentración municipal y hosp. por municipio (H2) | Mortalidad (`FEC_DEF` nulo) |
| Perfil edad/sexo/gestantes dentro del Valle (H3) | Regresión municipio–semana |

## 1.4. Justificación del uso de IA / Ciencia de Datos

| Técnica | Aplicación en este trabajo | Pertinencia |
|---|---|---|
| Perfilamiento de datos | Dimensiones, tipos, nulos y duplicados | Verifica calidad antes de interpretar |
| Limpieza y transformación | Fechas, `edad_anios`, flags clínicos, filtro Valle | Indicadores consistentes en un solo universo |
| Analítica descriptiva | Conteos, %, medias y medianas | Cuantifica carga y focos |
| Análisis temporal | Mes y semana epidemiológica **en Valle** | Identifica la ventana de pico (H1) |
| Segmentación municipal | Ranking y share acumulado | Responde la meta del 60 % (H2 / SMART) |
| Poblaciones especiales | Sexo, grupos de edad, gestantes | Responde H3 y la priorización clínica |
| Visualización | Tablas y gráficas en Jupyter | Comunicación para vigilancia |

La **IA generativa** se usó para estructurar el notebook, adaptar el Taller 1 al alcance Valle 2025, proponer código y organizar interpretaciones. No reemplaza la validación humana: se verifican filtros territoriales, fórmulas de KPI y coherencia con SIVIGILA.

## 1.5. Hipótesis de trabajo

| ID | Hipótesis | Variables involucradas |
|----|-----------|------------------------|
| **H1** | En el Valle 2025 hay **estacionalidad**: los casos se concentran en ciertos meses/semanas epidemiológicas (ventana de pico). | `SEMANA`, `FEC_NOT`, `mes`, `AREA` |
| **H2** | Hay **heterogeneidad municipal**: pocos municipios concentran ≥ 60 % de los casos y la hospitalización no es uniforme entre focos. | `Municipio_ocurrencia`, `hospitalizado` |
| **H3** | El **perfil demográfico y de poblaciones** (edad, sexo, gestantes) y la hospitalización **difieren entre municipios-foco** y grupos de edad. | `edad_anios`, `SEXO`, `GP_GESTAN`, `sem_ges`, `hospitalizado`, `Municipio_ocurrencia` |

## 1.6. Alcance de este EDA (Taller 2)

- **Fuente única:** `Datos_2025_210.xlsx` (evento 210 — Dengue, año **2025**)
- **Población / territorio:** notificaciones con **departamento de ocurrencia = Valle del Cauca** (focos epidemiológicos). No analizamos el ranking nacional de departamentos.
- **Poblaciones dentro del Valle:** sexo, grupos de edad y **gestantes** (`GP_GESTAN`, `sem_ges`)
- **No analizamos** otros años, IDEAM/DANE, ni mortalidad
- Diccionario, limpieza y conclusiones viven **en este notebook**


---
# 2. Propuesta de arquitectura de datos

> *Etapa del flujo: **Arquitectura (obtención → almacenes → ETL → tipos → muestra)***
>
> Entregable de la rúbrica del Taller 2 (5 pts): cómo se obtienen los datos, almacenes, ETL/ELT, tipos y muestra.
> Documento PDF complementario: [`docs/Propuesta_arquitectura_datos_dengue_taller2.pdf`](../docs/Propuesta_arquitectura_datos_dengue_taller2.pdf).

Esta sección baja el alcance de la propuesta previa (Valle 2023 / visión nacional) al universo del notebook: **Valle del Cauca, año 2025**, focos municipales y poblaciones (edad, sexo, gestantes).

## 2.1. Objetivo que orienta la arquitectura

> En el Valle del Cauca durante 2025, identificar si el **top 10 de municipios** concentra ≥ **60 %** de las notificaciones de dengue, con una **ventana de pico** (mes/semana) y un perfil demográfico/clínico (edad, sexo, % hospitalización, gestantes) que permita priorizar focos municipales.

## 2.2. Recorrido de los datos (flujo por lotes)

```
SIVIGILA evento 210          data/raw/                 ETL Python                 data/processed/              Consumo
Datos_2025_210.xlsx  →  Excel (almacén original)  →  fechas · edad · filtro  →  Parquet Valle limpio  →  Notebook EDA
(~120.5k × ~70)             + registro procedencia     Valle + FEC_NOT 2025     dengue_sivigila_2025_      (+ Streamlit opcional)
                                                       + features derivadas      valle_limpio.parquet
                                                       + controles calidad       → tablas agregadas
```

| Paso | Qué ocurre | Salida |
|------|------------|--------|
| **1. Fuente** | Portal INS / SIVIGILA, evento 210 — Dengue, archivo 2025 | `Datos_2025_210.xlsx` |
| **2. Almacén original** | Copia íntegra en `data/raw/`; no se edita a mano | Excel crudo |
| **3. ETL (no ELT)** | Extraer → transformar en Python → cargar almacén analítico | Parquet Valle + tablas |
| **4. Almacén procesado** | Solo ocurrencia Valle + `FEC_NOT` 2025 + features | `dengue_sivigila_2025_valle_limpio.parquet` |
| **5. Resultados** | Univariado/bivariado H1–H3; KPI SMART | Este notebook |

**Por qué ETL y no ELT:** transformamos (fechas, edad, filtros, flags) **antes** de materializar el Parquet analítico. No cargamos el Excel crudo a un warehouse para transformar después.

## 2.3. Obtención y almacenes

| Almacén / formato | Contenido y uso | Referencia |
|-------------------|-----------------|------------|
| **Original — Excel** | Fuente íntegra en `data/raw/`; registrar portal, fecha y versión | `Datos_2025_210.xlsx` (~120.5k × ~70) |
| **Procesado nacional — Parquet** | Limpieza nacional del Taller 1 (opcional; **no** es el universo Taller 2) | `dengue_sivigila_2025_limpio.parquet` |
| **Procesado analítico — Parquet Valle** | Universo del EDA: Valle 2025 + tipos + features | `dengue_sivigila_2025_valle_limpio.parquet` |
| **Resultados** | Agregados municipio / mes / semana / sexo / grupo_edad / gestantes | Tablas y gráficas de §§ 7–8 |

El parquet del Valle **no pisa** el del Taller 1.

## 2.4. ETL: extraer, transformar y cargar

| Fase | Acciones |
|------|----------|
| **Extraer** | `load_dengue(mode="year", year=2025)` desde `data/raw/` |
| **Transformar** | Parsear fechas; filtrar `FEC_NOT` ≠ 2025; `edad_anios`; `mes`, `hospitalizado`, `confirmado`, `gestante`, `grupo_edad`; filtrar `Departamento_ocurrencia` ≈ Valle; drop columnas ~100 % nulas; dropna en `COD_MUN_O` / `FEC_NOT` |
| **Cargar** | Escribir Parquet Valle; alimentar análisis y (opcional) Streamlit |

**Controles de calidad:** nulos estructurales (`FEC_DEF`, `sem_ges` fuera de gestantes, `FEC_HOS`); coherencia temporal; conteo de exclusiones; unicidad de `CONSECUTIVE`.

## 2.5. Tipos de datos y tratamiento

| Campo(s) | Tipo | Tratamiento |
|----------|------|-------------|
| `CONSECUTIVE` | Identificador | Conservar; revisar unicidad |
| `FEC_NOT`, `INI_SIN`, `FEC_HOS` | Temporal | `datetime`; acotar `FEC_NOT` a 2025 |
| `EDAD` + `UNI_MED`; `SEMANA`; `ANO` | Numérico | `edad_anios`; semana/año enteros |
| `Departamento_ocurrencia`, `Municipio_ocurrencia` | Categórico | Filtro Valle; municipio = foco (H2) |
| `SEXO`, `AREA`, `grupo_edad` | Categórico | Perfil poblacional (H3) |
| `PAC_HOS`, `confirmados`, `GP_GESTAN` | Categórico codificado | Flags `hospitalizado` / `confirmado` / `gestante` |
| `sem_ges` | Numérico | Solo gestantes; nulo estructural en el resto |

## 2.6. Muestra de entrada (antes del filtro Valle)

Cada fila es una notificación. Códigos: `PAC_HOS` 1=Sí / 2=No; `confirmados` 1=Sí.

| CONSECUTIVE | FEC_NOT | Depto. ocurrencia | Municipio | SEXO | PAC_HOS | confirmados |
|-------------|---------|-------------------|-----------|------|---------|-------------|
| 23659 | 2025-03-30 | ARAÚCA | … | F | 2 | 1 |
| … | 2025-01-15 | VALLE | CALI | M | 1 | 1 |
| … | 2025-02-03 | VALLE | PALMIRA | F | 2 | 1 |
| … | 2025-01-22 | VALLE | BUENAVENTURA | M | 2 | 0 |
| … | 2025-04-10 | VALLE | TULUÁ | F | 1 | 1 |

Tras el ETL quedan solo filas Valle + `FEC_NOT` 2025 (+ features). La muestra real se inspecciona con `df.head()` / `df_limpio.head()` en §§ 5–6.

## 2.7. Salida esperada

- Ranking municipal: casos, % Valle, % acumulado, % hospitalización, edad mediana, n gestantes  
- Series mes / semana epidemiológica (H1)  
- Perfil sexo, `grupo_edad`, gestantes vs otras mujeres (H3)  
- Meta SMART: share top 10 ≥ **60 %**

$$
\text{Proporción de hospitalización (\%)} = 100 \times \frac{\text{casos hospitalizados}}{\text{total de casos del nivel}}
$$

## 2.8. Alcance y fuera de alcance

| Dentro | Fuera |
|--------|-------|
| Notificaciones SIVIGILA 2025, Valle (ocurrencia) | Otros años / otros departamentos como universo |
| Hospitalización, confirmación, demografía, gestantes | Mortalidad (`FEC_DEF` 100 % nulo) |
| Priorización municipal descriptiva | Modelo predictivo, IDEAM, DANE |

PDF detallado (misma propuesta en formato de entrega Moodle): **`docs/Propuesta_arquitectura_datos_dengue_taller2.pdf`**.


---
# 3. Carga del conjunto de datos

> *Etapa del flujo: **Carga de datos***

### Fuente utilizada en este EDA

| Fuente | Archivo | Estado |
|--------|---------|--------|
| **INS / SIVIGILA** (evento 210 — Dengue) | `Datos_2025_210.xlsx` | ✅ Se carga el archivo y se **acota** a Valle del Cauca |

> Carga con utilidades del proyecto (`load_dengue`, modo `year`, `ANIO = 2025`). El recorte territorial se aplica **después** de parsear fechas, para no mezclar registros de otros años.


In [ ]:
# Importación de librerías
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Permite importar desde src/ al ejecutar el notebook
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_RAW, DATA_PROCESSED
from src.load_data import load_dengue

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 13
pd.set_option("display.max_columns", None)

archivo_2025 = DATA_RAW / "Datos_2025_210.xlsx"
print(f"Proyecto: {PROJECT_ROOT}")
print(f"Fuente del EDA: {archivo_2025}")
print(f"Existe: {archivo_2025.exists()}")


In [ ]:
# Carga del Excel SIVIGILA 2025 y recuento del recorte territorial
MODO = "year"
ANIO = 2025
DEPTO_FOCO = "VALLE"

df = load_dengue(MODO, year=ANIO)

print(f"Fuente: Datos_{ANIO}_210.xlsx")
print(f"Dimensiones crudas: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Memoria aprox.: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"COD_EVE: {df['COD_EVE'].unique()}")
print(f"Año en df: {sorted(df['ANO'].unique())}")

mask_ocurrencia = df["Departamento_ocurrencia"].astype(str).str.contains(DEPTO_FOCO, case=False, na=False)
mask_residencia = df["Departamento_residencia"].astype(str).str.contains(DEPTO_FOCO, case=False, na=False)
print()
print("--- Recorte de alcance (antes de limpieza) ---")
print(f"Ocurrencia en Valle:  {mask_ocurrencia.sum():,} ({mask_ocurrencia.mean()*100:.1f}% del archivo)")
print(f"Residencia en Valle:  {mask_residencia.sum():,} ({mask_residencia.mean()*100:.1f}% del archivo)")
print("Universo de este taller: ocurrencia en Valle (focos epidemiológicos).")


---
# 4. Diccionario de datos

> *Etapa del flujo: **Diccionario de datos***

Documentamos el **significado, tipo, relevancia y expectativa** de las variables clave para el Valle 2025.

Para cada variable:

- **Variable:** nombre de la columna
- **Tipo:** numérica, categórica, fecha
- **Descripción:** qué representa
- **Relevancia:** objetivo / predictora / contexto / descartar
- **Expectativa:** utilidad para priorizar focos/poblaciones (**Alta / Media / Baja**)

## 4.1. Diccionario del conjunto de datos SIVIGILA (evento 210)

### Variables objetivo y de desenlace

| Variable | Tipo | Descripción | Relevancia | Expectativa |
|----------|------|-------------|------------|-------------|
| `CONSECUTIVE` | int | Identificador único del registro | Clave primaria | Baja |
| `confirmados` | categórico | Caso confirmado (1=Sí, 0=No) | Filtro de casos | Alta |
| `CON_FIN` / `Estado_final_de_caso` | categórico | Condición final del paciente | Desenlace clínico | Media |
| `FEC_DEF` | fecha | Fecha de defunción | Mortalidad (100 % nulos → se descarta) | Baja |
| `PAC_HOS` | categórico | Hospitalizado (1=Sí, 2=No) | Severidad / carga clínica | Alta |
| `nom_est_f_caso` | texto | Estado final (lab, probable, etc.) | Clasificación | Media |

### Variables temporales

| Variable | Tipo | Descripción | Relevancia | Expectativa |
|----------|------|-------------|------------|-------------|
| `ANO` | int (temporal) | Año de notificación | Serie temporal (fijo 2025) | Media |
| `SEMANA` | int (temporal) | Semana epidemiológica | Agregación temporal (H1) | Alta |
| `FEC_NOT` | texto/fecha | Fecha de notificación | Convertir a datetime; base de `mes` | Alta |
| `INI_SIN` | texto/fecha | Inicio de síntomas | Latencia del brote | Media |

### Variables territoriales (alcance Valle)

| Variable | Tipo | Descripción | Relevancia | Expectativa |
|----------|------|-------------|------------|-------------|
| `COD_DPTO_O` / `Departamento_ocurrencia` | int/texto | Departamento de ocurrencia | **Filtro de universo** (Valle) | Alta |
| `COD_MUN_O` / `Municipio_ocurrencia` | int/texto | Municipio de ocurrencia | Unidad de foco (H2) | Alta |
| `Departamento_residencia` / `Municipio_residencia` | texto | Residencia del paciente | Contexto (puede diferir de ocurrencia) | Media |
| `AREA` | categórico | Urbana / Rural / Urbana-rural (1/2/3) | Contexto | Media |

### Variables demográficas y poblaciones

| Variable | Tipo | Descripción | Relevancia | Expectativa |
|----------|------|-------------|------------|-------------|
| `EDAD` | int | Edad (unidad según `UNI_MED`) | Perfil epidemiológico | Media |
| `UNI_MED` | categórico | 1=años, 2=meses, 3=días | Normalizar a `edad_anios` | Alta |
| `SEXO` | texto | Sexo (M/F) | Perfil (H3) | Media |
| `GP_GESTAN` | categórico | Gestante (1=Sí, 2=No) | Población especial (H3) | Alta |
| `sem_ges` | numérica | Semanas de gestación | Solo gestantes; nulo estructural en el resto | Media |
| `TIP_CAS` | categórico | Tipo de caso (2=Probable, 3=Confirmado) | Clasificación | Media |

### Features derivadas (se crean en §5.4)

| Variable | Tipo | Descripción | Relevancia | Expectativa |
|----------|------|-------------|------------|-------------|
| `edad_anios` | numérica | Edad en años | Perfil y grupos etarios | Alta |
| `mes` / `trimestre` | int | Extraídos de `FEC_NOT` | Estacionalidad | Alta |
| `hospitalizado` / `confirmado` | binaria | Flags 0/1 | KPI clínicos | Alta |
| `gestante` | binaria | `GP_GESTAN == 1` | Población especial | Alta |
| `grupo_edad` | categórica | 0–4, 5–17, 18–44, 45–59, 60+ | Segmentación poblacional | Alta |


In [ ]:
# Diccionario de variables clave como DataFrame (guía del profesor)
columnas_clave = [
    "CONSECUTIVE", "ANO", "SEMANA", "FEC_NOT", "EDAD", "UNI_MED", "SEXO",
    "COD_DPTO_O", "Departamento_ocurrencia", "COD_MUN_O", "Municipio_ocurrencia",
    "confirmados", "PAC_HOS", "GP_GESTAN", "sem_ges", "AREA", "nom_est_f_caso",
]

diccionario = pd.DataFrame({
    "variable": columnas_clave,
    "tipo": [
        "int", "int (temporal)", "int (temporal)", "texto/fecha", "int", "categórico", "categórico",
        "int", "categórico", "int", "categórico",
        "categórico", "categórico", "categórico", "numérica (estructuralmente nula)",
        "categórico", "categórico",
    ],
    "descripcion": [
        "Identificador único del registro",
        "Año de notificación",
        "Semana epidemiológica",
        "Fecha de notificación del caso",
        "Edad del paciente (según UNI_MED)",
        "Unidad de edad (1=años, 2=meses, 3=días)",
        "Sexo del paciente (M/F)",
        "Código DANE del departamento de ocurrencia",
        "Nombre del departamento de ocurrencia (filtro Valle)",
        "Código DANE del municipio de ocurrencia",
        "Nombre del municipio de ocurrencia (foco)",
        "Caso confirmado (1=Sí, 0=No)",
        "Hospitalizado (1=Sí, 2=No)",
        "Gestante (1=Sí, 2=No)",
        "Semanas de gestación (aplica si GP_GESTAN=1)",
        "Área (1=Urbana, 2=Rural, 3=Urbana-rural)",
        "Estado final del caso (lab, probable, etc.)",
    ],
    "relevancia": [
        "Clave primaria", "Serie temporal", "Agregación temporal", "Convertir a datetime",
        "Perfil epidemiológico", "Normalizar edad", "Perfil epidemiológico",
        "Filtro de universo", "Filtro de universo", "Unidad de foco", "Reportes municipales",
        "Filtro de casos", "Severidad", "Población especial", "Clínica obstétrica",
        "Contexto", "Clasificación",
    ],
    "expectativa": [
        "Baja", "Media", "Alta", "Alta",
        "Media", "Alta", "Media",
        "Alta", "Alta", "Alta", "Alta",
        "Alta", "Alta", "Alta", "Media",
        "Media", "Media",
    ],
})

diccionario


---
# 5. Visión general del conjunto de datos

> *Etapa del flujo: **Visión general (formato, nombres, tipos, unidades, columnas redundantes)***

Primero inspeccionamos el **archivo nacional** (misma fuente del Taller 1) y luego el recorte a Valle. El análisis univariado/bivariado se hace solo sobre el universo Valle 2025.

## 5.1. Primeras filas del conjunto de datos


In [ ]:
# El método head() permite una inspección visual rápida de los registros.
df.head()


## 5.2. Estructura, tipos de datos y memoria


In [ ]:
# info() resume registros no nulos y tipo de dato por columna.
df.info()


## 5.3. Dimensiones, columnas y valores únicos


In [ ]:
print("Filas:", df.shape[0], "| Columnas:", df.shape[1])
print("-" * 60)

# Cantidad de valores distintos por columna: ayuda a distinguir
# variables categóricas (pocos valores) de continuas (muchos valores).
df.nunique().sort_values()


## 5.4. Estadísticos descriptivos preliminares

`EDAD` aún es cruda (depende de `UNI_MED`). La interpretación de media/mediana espera a `edad_anios` en §5.4.


In [ ]:
# Estadísticos preliminares: SEMANA es usable; EDAD es cruda
cols_describe = ["EDAD", "SEMANA"]
print("Nota: EDAD aún no está normalizada; no interpretar media/mediana hasta crear edad_anios.")
df[cols_describe].describe().T


In [ ]:
# Resumen de variables categóricas clave (incluye códigos enteros)
cols_cat = [
    "SEXO", "PAC_HOS", "confirmados", "TIP_CAS", "AREA",
    "nom_est_f_caso", "Departamento_ocurrencia", "GP_GESTAN",
]
for col in cols_cat:
    if col in df.columns:
        print(f"\n--- {col} (top 5) ---")
        print(df[col].value_counts().head())


---
# 6. Limpieza y preparación de los datos

> *Etapa del flujo: preparación previa al análisis (valores perdidos, formato)*

Limpiamos el archivo 2025 y **después** acotamos a ocurrencia en Valle. Así el filtro de `FEC_NOT` al año y las features derivadas quedan consistentes.

## 6.1. Identificación de valores perdidos

### Paso 1 — Conteo de nulos por columna


In [ ]:
nulos = df.isnull().sum()
nulos[nulos > 0].sort_values(ascending=False)


### Paso 2 — Porcentaje de valores perdidos por columna


In [ ]:
pct_nulos = (df.isnull().mean() * 100).round(2)
pct_nulos[pct_nulos > 0].sort_values(ascending=False)


## 6.2. Tratamiento de los valores perdidos

Tabla de justificación (estilo guía del profesor):

| Columna | Faltantes / hallazgo | Estrategia | Justificación |
|---------|----------------------|------------|---------------|
| `GRU_POB`, `CBMTE`, `FM_FUERZA`, `FM_UNIDAD`, `FM_GRADO` | ~100 % nulos | **Drop columna** | Sin información útil para el EDA |
| `FEC_DEF` | 100 % nulos en 2025 | **Drop columna** | No permite analizar mortalidad |
| `COD_ASE` | Alta proporción de nulos / poco usable | **Drop columna** | No aporta al análisis municipal ni clínico |
| `FEC_HOS` | Nulos frecuentes | **Conservar** | Nulo = no hospitalizado (esperado); no imputamos |
| `sem_ges` | Vacío en no gestantes | **Conservar** | Nulo estructural; se analiza solo en gestantes |
| `COD_MUN_O`, `FEC_NOT` | Críticos si faltan | **Dropna filas** | Sin municipio o fecha no hay H1/H2 |
| Resto (`SEXO`, `EDAD`, `PAC_HOS`, `GP_GESTAN`) | Bien pobladas | **Conservar** | Base del perfil demográfico y poblaciones |


In [ ]:
df_limpio = df.copy()

# Drop: columnas vacías / sin utilidad
cols_vacias = ["GRU_POB", "CBMTE", "FEC_DEF", "FM_FUERZA", "FM_UNIDAD", "FM_GRADO", "COD_ASE"]
df_limpio = df_limpio.drop(columns=[c for c in cols_vacias if c in df_limpio.columns])

# Dropna: filas sin municipio o fecha de notificación (críticos para H1/H2)
filas_antes = len(df_limpio)
df_limpio = df_limpio.dropna(subset=["COD_MUN_O", "FEC_NOT"])
print(f"Columnas eliminadas: {cols_vacias}")
print(f"Filas eliminadas: {filas_antes - len(df_limpio):,}")
print(f"Filas restantes: {len(df_limpio):,}")


### Verificación: estado de valores perdidos tras limpieza


In [ ]:
# Top columnas con nulos restantes
resto = df_limpio.isnull().sum()
resto[resto > 0].sort_values(ascending=False).head(10)


## 6.3. Corrección del formato de los datos

Convertimos fechas en texto a `datetime` con formato explícito (`dd/mm/yyyy HH:MM:SS`).

Filtramos registros cuya `FEC_NOT` cae fuera de **2025**. El EDA temporal de este taller se acota al año del Excel.


In [ ]:
def parsear_fecha(serie):
    """Convierte fechas SIVIGILA (dd/mm/yyyy con 'a. m.' / 'p. m.') a datetime."""
    limpia = (
        serie.astype(str)
        .str.replace("\xa0", " ", regex=False)
        .str.replace(r"\s*a\.\s*m\.", "", regex=True)
        .str.replace(r"\s*p\.\s*m\.", "", regex=True)
        .str.strip()
    )
    dt = pd.to_datetime(limpia, format="%d/%m/%Y %H:%M:%S", errors="coerce")
    mask = dt.isna() & limpia.ne("nan") & limpia.ne("NaT") & limpia.ne("") & limpia.ne("None")
    if mask.any():
        dt = dt.copy()
        dt.loc[mask] = pd.to_datetime(limpia.loc[mask], format="%d/%m/%Y", errors="coerce")
    return dt

cols_fecha = ["FEC_NOT", "INI_SIN", "FEC_HOS", "FEC_CON", "FECHA_NTO"]
for col in cols_fecha:
    if col in df_limpio.columns:
        df_limpio[col] = parsear_fecha(df_limpio[col])

print("FEC_NOT convertida:", df_limpio["FEC_NOT"].dtype)
print("Rango crudo:", df_limpio["FEC_NOT"].min(), "→", df_limpio["FEC_NOT"].max())

fuera_anio = df_limpio["FEC_NOT"].notna() & (df_limpio["FEC_NOT"].dt.year != ANIO)
n_fuera = int(fuera_anio.sum())
print(f"Registros con FEC_NOT fuera de {ANIO}: {n_fuera:,} ({n_fuera/len(df_limpio)*100:.2f}%)")
if n_fuera > 0:
    print(df_limpio.loc[fuera_anio, "FEC_NOT"].dt.year.value_counts().sort_index())

filas_antes = len(df_limpio)
df_limpio = df_limpio.loc[~fuera_anio].copy()
print(f"Filas filtradas (FEC_NOT fuera de {ANIO}): {filas_antes - len(df_limpio):,}")
print(f"Filas restantes (nacional {ANIO}): {len(df_limpio):,}")
print("Rango final FEC_NOT:", df_limpio["FEC_NOT"].min(), "→", df_limpio["FEC_NOT"].max())
df_limpio[["FEC_NOT", "INI_SIN", "FEC_HOS"]].head()


## 6.4. Creación de nuevas columnas (features derivadas) y recorte a Valle del Cauca


In [ ]:
def edad_en_anios(row):
    if row["UNI_MED"] == 1:
        return row["EDAD"]
    if row["UNI_MED"] == 2:
        return row["EDAD"] / 12
    return row["EDAD"] / 365

df_limpio["edad_anios"] = df_limpio.apply(edad_en_anios, axis=1)
df_limpio["mes"] = df_limpio["FEC_NOT"].dt.month
df_limpio["trimestre"] = df_limpio["FEC_NOT"].dt.quarter
df_limpio["hospitalizado"] = df_limpio["PAC_HOS"] == 1
df_limpio["confirmado"] = df_limpio["confirmados"] == 1
df_limpio["gestante"] = df_limpio["GP_GESTAN"] == 1 if "GP_GESTAN" in df_limpio.columns else False
df_limpio["sem_ges"] = pd.to_numeric(df_limpio.get("sem_ges"), errors="coerce")
df_limpio["depto_mun"] = (
    df_limpio["COD_DPTO_O"].astype(str).str.zfill(2)
    + df_limpio["COD_MUN_O"].astype(str).str.zfill(3)
)
df_limpio["grupo_edad"] = pd.cut(
    df_limpio["edad_anios"],
    bins=[0, 5, 18, 45, 60, 200],
    labels=["0-4", "5-17", "18-44", "45-59", "60+"],
    right=False,
)

# Recorte de alcance: ocurrencia en Valle del Cauca
n_nacional = len(df_limpio)
mask_valle = df_limpio["Departamento_ocurrencia"].astype(str).str.contains(
    DEPTO_FOCO, case=False, na=False
)
df_nacional = df_limpio.copy()
df_limpio = df_limpio.loc[mask_valle].copy()

print(f"Casos nacionales {ANIO} (tras filtro FEC_NOT): {n_nacional:,}")
print(f"Casos VALLE (ocurrencia): {len(df_limpio):,} ({len(df_limpio)/n_nacional*100:.1f}% del país)")
print(f"Municipios de ocurrencia en Valle: {df_limpio['Municipio_ocurrencia'].nunique()}")
print()
df_limpio[
    ["EDAD", "UNI_MED", "edad_anios", "grupo_edad", "mes", "hospitalizado", "confirmado", "gestante", "Municipio_ocurrencia"]
].head()


In [ ]:
# Exportar dataset limpio del Valle 2025 (no pisa el parquet nacional del Taller 1)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
ruta_limpio = DATA_PROCESSED / f"dengue_sivigila_{ANIO}_valle_limpio.parquet"
df_limpio.to_parquet(ruta_limpio, index=False)
print(f"Exportado: {ruta_limpio}")
print(f"Filas: {len(df_limpio):,} | Columnas: {df_limpio.shape[1]}")


Con esto concluye la etapa de limpieza. El conjunto de trabajo es **Valle del Cauca 2025**: tipos correctos, features derivadas y universo territorial acotado.

---
# 7. Análisis univariado

> *Etapa del flujo: **Análisis univariado***

## 7.1. Variables objetivo: casos, hospitalización y perfil temporal (Valle)

**Preguntas guía** (se responden al ejecutar las celdas; interpretación en la markdown siguiente):

| Pregunta guía | Dónde se responde | Hipótesis / KPI |
|----------------|-------------------|-----------------|
| ¿Cómo se distribuyen los casos por mes y semana en el Valle? | Gráficos de `mes` y `SEMANA` | **H1** (estacionalidad) |
| ¿Cuántas hospitalizaciones hay? | Resumen `hospitalizado` | KPI clínico / SMART |
| ¿Qué proporción son confirmados? | `confirmado` / `nom_est_f_caso` | KPI de clasificación |
| ¿Cuántos casos son gestantes? | `gestante` | Población especial (H3) |


In [ ]:
# Resumen de variables de desenlace — Valle 2025
print(f"Total casos Valle {ANIO}:", f"{len(df_limpio):,}")
print("Casos confirmados:", int(df_limpio["confirmado"].sum()), f"({df_limpio['confirmado'].mean()*100:.1f}%)")
print("Hospitalizados:", int(df_limpio["hospitalizado"].sum()), f"({df_limpio['hospitalizado'].mean()*100:.1f}%)")
print("Gestantes:", int(df_limpio["gestante"].sum()), f"({df_limpio['gestante'].mean()*100:.2f}%)")
print()
print("Estado final del caso:")
print(df_limpio["nom_est_f_caso"].value_counts())


In [ ]:
# Casos por mes — Valle 2025
casos_por_mes = df_limpio.groupby("mes").size()
ax = casos_por_mes.plot(
    kind="bar", color="crimson", title=f"Casos de dengue por mes — Valle del Cauca {ANIO}"
)
plt.xlabel("Mes")
plt.ylabel("Número de casos")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
casos_por_mes


In [ ]:
# Casos por semana epidemiológica — Valle 2025
casos_por_semana = df_limpio.groupby("SEMANA").size()
casos_por_semana.plot(
    color="crimson", marker="o", title=f"Casos por semana epidemiológica — Valle del Cauca {ANIO}"
)
plt.xlabel("Semana epidemiológica")
plt.ylabel("Número de casos")
plt.tight_layout()
plt.show()
print("Semana de máximo:", int(casos_por_semana.idxmax()), "con", int(casos_por_semana.max()), "casos")
print("Top 5 semanas:")
print(casos_por_semana.sort_values(ascending=False).head())


**Interpretación §7.1 (Valle 2025) — respuestas a las preguntas guía:**

Ejecutar las celdas de arriba imprime los KPI exactos. Lectura esperada (alineada con Taller 1, ahora **solo Valle**):

| Pregunta | Qué buscar en los gráficos | Hipótesis |
|----------|----------------------------|-----------|
| ¿Cómo se distribuyen mes/semana? | Pico al **inicio del año** (ene–feb / primeras semanas) y descenso posterior | **H1** si hay ventana de pico clara |
| ¿Cuántas hospitalizaciones? | % hosp. del Valle (en Taller 1 era ~26.5 %, menor que el ~37 % nacional) | KPI clínico |
| ¿Confirmados? | Proporción `confirmado` | KPI de clasificación |
| ¿Gestantes? | Conteo y % sobre el total Valle | Insumo de **H3** |

**Calidad temporal:** el filtro de `FEC_NOT` fuera de 2025 se aplicó **antes** del recorte territorial, para no contaminar la estacionalidad.


## 7.2. Variables numéricas continuas

Variables: `edad_anios`, `SEMANA`, y `sem_ges` **solo en gestantes**.

**Preguntas guía:**
- ¿Cómo se distribuye la edad de los casos en el Valle? → histograma/boxplot. Alimenta **H3**.
- ¿Las semanas de gestación cubren todo el embarazo o se concentran en un trimestre? → `sem_ges` en gestantes.
- ¿Hay valores extremos en edad o semana? → §6.2.1 (IQR). No eliminamos outliers automáticamente.


In [ ]:
def analizar_variable_continua(columna, data=None, color="teal"):
    """Muestra estadísticos, histograma y boxplot de una variable continua."""
    data = df_limpio if data is None else data
    datos = data[columna].dropna()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(datos, bins=30, kde=True, ax=axes[0], color=color)
    axes[0].set_title(f"Histograma — {columna}")
    axes[0].set_xlabel(columna)
    sns.boxplot(x=datos, ax=axes[1], color=color)
    axes[1].set_title(f"Boxplot — {columna}")
    plt.tight_layout()
    plt.show()
    display(datos.describe().round(2))

print("===== edad_anios (todos los casos del Valle) =====")
analizar_variable_continua("edad_anios", color="crimson")

print("===== SEMANA =====")
analizar_variable_continua("SEMANA", color="steelblue")

df_gestantes = df_limpio.loc[df_limpio["gestante"]].copy()
print(f"===== sem_ges (solo gestantes, n={len(df_gestantes):,}) =====")
if df_gestantes["sem_ges"].notna().any():
    analizar_variable_continua("sem_ges", data=df_gestantes, color="teal")
else:
    print("No hay valores numéricos de sem_ges en gestantes.")


### 7.2.1. Cuantificación de outliers (IQR)

> La detección de outliers **no implica** eliminación automática. Documentar y justificar.


In [ ]:
def contar_outliers_iqr(columna, data=None):
    data = df_limpio if data is None else data
    q1, q3 = data[columna].quantile([0.25, 0.75])
    iqr = q3 - q1
    mask = (data[columna] < q1 - 1.5 * iqr) | (data[columna] > q3 + 1.5 * iqr)
    n = int(mask.sum())
    return n, len(data), n / len(data) * 100

for col in ["edad_anios", "SEMANA"]:
    n, total, pct = contar_outliers_iqr(col)
    print(f"{col}: {n:,} outliers ({pct:.2f}% de {total:,} registros)")

print("\n--- Revisión de extremos en edad_anios ---")
print("Mínimos (posibles neonatos):")
display(df_limpio.nsmallest(10, "edad_anios")[["EDAD", "UNI_MED", "edad_anios", "Municipio_ocurrencia"]])

print("Máximos (edad avanzada):")
display(df_limpio.nlargest(10, "edad_anios")[["EDAD", "UNI_MED", "edad_anios", "Municipio_ocurrencia"]])

print("Distribución de UNI_MED (1=años, 2=meses, 3=días):")
print(df_limpio["UNI_MED"].value_counts().sort_index())

print("\nChequeos rápidos:")
print("< 1 año:", int((df_limpio["edad_anios"] < 1).sum()))
print(">= 90 años:", int((df_limpio["edad_anios"] >= 90).sum()))
print(">= 100 años:", int((df_limpio["edad_anios"] >= 100).sum()))


## 7.3. Variables categóricas

Variables: `SEXO`, `AREA`, `grupo_edad`, `Municipio_ocurrencia`.

**Preguntas guía:**
- ¿Cómo se reparte sexo, área y grupo de edad en el Valle?
- ¿Qué municipios concentran más casos? → top 15. Responde **H2** (univariado).


In [ ]:
MAPEO_CATEGORIAS = {
    "SEXO": {"M": "M — Masculino", "F": "F — Femenino"},
    "AREA": {1: "1 — Urbana", 2: "2 — Rural", 3: "3 — Urbana-rural"},
}

def analizar_variable_categorica(columna, data=None, top_n=15, mapeo=None):
    data = df_limpio if data is None else data
    serie = data[columna]
    if mapeo is not None:
        serie = serie.map(mapeo).fillna(serie.astype(str))
    freq = serie.value_counts().head(top_n)
    freq.plot(kind="barh", color="crimson", title=f"Frecuencia — {columna} (Valle {ANIO})")
    plt.xlabel("Conteo")
    plt.tight_layout()
    plt.show()
    display(freq.to_frame("conteo"))

for col in ["SEXO", "AREA", "grupo_edad"]:
    analizar_variable_categorica(col, mapeo=MAPEO_CATEGORIAS.get(col))


In [ ]:
# Top 15 municipios del Valle por número de casos (focos)
top_mun = df_limpio["Municipio_ocurrencia"].value_counts()
share_top10 = top_mun.head(10).sum() / len(df_limpio) * 100
share_top5 = top_mun.head(5).sum() / len(df_limpio) * 100

print(f"Municipios con al menos 1 caso: {top_mun.shape[0]}")
print(f"Share top 5:  {share_top5:.1f}%")
print(f"Share top 10: {share_top10:.1f}%  ← meta SMART ≥ 60 %")

fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(
    x=top_mun.head(15).values,
    y=top_mun.head(15).index,
    hue=top_mun.head(15).index,
    palette="Reds_r",
    legend=False,
    ax=ax,
)
ax.set_title(f"Focos epidemiológicos: top 15 municipios del Valle — {ANIO}")
ax.set_xlabel("Número de casos")
ax.set_ylabel("Municipio")
for p in ax.patches:
    width = p.get_width()
    ax.annotate(f"{int(width):,}", (width, p.get_y() + p.get_height() / 2),
                ha="left", va="center", xytext=(5, 0), textcoords="offset points")
plt.tight_layout()
plt.show()

top10_tabla = pd.DataFrame({
    "Casos": top_mun.head(10),
    "% Valle": (top_mun.head(10) / len(df_limpio) * 100).round(2),
    "% acumulado": (top_mun.head(10).cumsum() / len(df_limpio) * 100).round(2),
})
top10_tabla


**Interpretación §§7.2–7.3 (Valle 2025):**

| Hallazgo | Qué confirma |
|----------|----------------|
| Edad: en Taller 1, Valle tenía media ~28 y mediana ~24 (más alta que el país) | Base demográfica de **H3** |
| Sexo relativamente equilibrado | Contrasta con el leve predominio masculino nacional del Taller 1 |
| Área predominantemente urbana | La vigilancia se juega sobre todo en cabeceras |
| Top municipios y share del top 10 | **H2** (univariado): si share top 10 ≥ 60 %, se cumple el umbral SMART |

El detalle de hospitalización por municipio y de gestantes se cierra en el bivariado (§7).


---
# 8. Análisis bivariado

> *Etapa del flujo: **Análisis bivariado***

Cruzamos variables para **contrastar H1–H3** dentro del Valle (no vs el país).

## 8.1. Matriz de correlación (variables numéricas)

**Pregunta guía:** ¿Hay relaciones lineales útiles entre edad, semana y (en gestantes) semanas de gestación?  
**Respuesta esperada:** `SEMANA`–`mes` alta por construcción; edad casi no correlaciona con semana.


In [ ]:
# Correlación de variables numéricas/temporales en el Valle
cols_num = ["edad_anios", "SEMANA", "mes"]
corr = df_limpio[cols_num].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Matriz de correlación — edad y tiempo (Valle 2025)")
plt.tight_layout()
plt.show()

if df_limpio["gestante"].any():
    corr_g = df_limpio.loc[df_limpio["gestante"], ["edad_anios", "sem_ges", "SEMANA"]].corr()
    plt.figure(figsize=(6, 5))
    sns.heatmap(corr_g, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
    plt.title("Correlación en gestantes (edad, sem_ges, semana)")
    plt.tight_layout()
    plt.show()

print("Nota: PAC_HOS, confirmados y AREA son códigos categóricos; se analizan con tablas, no con Pearson.")


## 8.2. Diagramas de dispersión y carga semanal vs hospitalización

**Pregunta guía:** ¿La hospitalización se separa por edad y semana a nivel individual, o es mejor mirar **tasas agregadas**?


In [ ]:
# Dispersión: edad vs semana (muestra) y edad vs sem_ges en gestantes
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

muestra = df_limpio.dropna(subset=["edad_anios", "SEMANA"]).sample(
    n=min(4000, len(df_limpio)), random_state=42
)
sns.scatterplot(
    data=muestra, x="SEMANA", y="edad_anios", hue="hospitalizado",
    alpha=0.35, ax=axes[0], palette={False: "steelblue", True: "darkorange"},
)
axes[0].set_title("Edad vs semana (muestra) según hospitalización")
axes[0].legend(title="Hospitalizado", loc="upper right")

df_g = df_limpio.loc[df_limpio["gestante"]].dropna(subset=["edad_anios", "sem_ges"])
if len(df_g) > 0:
    sns.regplot(
        data=df_g, x="edad_anios", y="sem_ges", ax=axes[1],
        scatter_kws={"alpha": 0.45, "color": "teal"}, line_kws={"color": "darkred"},
    )
    r1 = df_g["edad_anios"].corr(df_g["sem_ges"])
    axes[1].set_title(f"Gestantes: edad vs semanas de gestación (r = {r1:.2f})")
else:
    axes[1].set_title("Sin gestantes con sem_ges numérica")
plt.tight_layout()
plt.show()

# Vista agregada: carga semanal vs % hospitalización
resumen_semana = (
    df_limpio.groupby("SEMANA")
    .agg(casos=("CONSECUTIVE", "count"), pct_hospitalizado=("hospitalizado", "mean"))
    .reset_index()
)
fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(resumen_semana["SEMANA"], resumen_semana["casos"], color="crimson", label="Casos")
ax1.set_xlabel("Semana epidemiológica")
ax1.set_ylabel("Casos", color="crimson")
ax1.tick_params(axis="y", labelcolor="crimson")

ax2 = ax1.twinx()
ax2.plot(
    resumen_semana["SEMANA"],
    resumen_semana["pct_hospitalizado"] * 100,
    color="darkorange",
    label="% hospitalización",
)
ax2.set_ylabel("% hospitalización", color="darkorange")
ax2.tick_params(axis="y", labelcolor="darkorange")
plt.title("Valle 2025 — carga semanal y % de hospitalización")
plt.tight_layout()
plt.show()


> **Lectura rápida:** a nivel individual la hospitalización **no se separa** bien por edad×semana (alta superposición). El % hosp. semanal puede **no bajar** cuando bajan los casos: la severidad observada no copia la curva de volumen. En gestantes, `sem_ges` vs edad suele ser débil (r bajo): no esperamos una relación lineal fuerte.

## 8.3. Casos por municipio y perfil clínico

**Preguntas guía (H2 / perfil clínico):**
- ¿El % de hospitalización cambia entre los municipios-foco?
- ¿Edad media y hospitalización difieren por sexo en el Valle?


In [ ]:
top_mun_nombres = df_limpio["Municipio_ocurrencia"].value_counts().head(10).index
df_top_mun = df_limpio[df_limpio["Municipio_ocurrencia"].isin(top_mun_nombres)].copy()

sns.countplot(
    data=df_top_mun, y="Municipio_ocurrencia", order=top_mun_nombres, color="crimson"
)
plt.title(f"Top 10 municipios del Valle por casos — {ANIO}")
plt.xlabel("Número de casos")
plt.tight_layout()
plt.show()

# Hospitalización por municipio (top 10)
tabla_hosp = pd.crosstab(
    df_top_mun["Municipio_ocurrencia"],
    df_top_mun["hospitalizado"],
    normalize="index",
)
tabla_hosp = tabla_hosp.reindex(top_mun_nombres)
tabla_hosp.columns = ["No hospitalizado", "Hospitalizado"]
tabla_hosp.plot(
    kind="bar", stacked=True,
    title="Proporción de hospitalización — top 10 municipios del Valle",
)
plt.ylabel("Proporción")
plt.xticks(rotation=45, ha="right")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()
display(tabla_hosp.round(3))


In [ ]:
# Edad media y % hospitalización / confirmados por sexo (Valle)
resumen_sexo = df_limpio.groupby("SEXO").agg(
    casos=("CONSECUTIVE", "count"),
    edad_media=("edad_anios", "mean"),
    edad_mediana=("edad_anios", "median"),
    pct_hospitalizado=("hospitalizado", "mean"),
    pct_confirmado=("confirmado", "mean"),
    n_gestantes=("gestante", "sum"),
).round(3)
resumen_sexo


> **Lectura rápida:** el ranking de barras confirma focos (Cali suele liderar). Las barras apiladas responden H2 clínico: **la hospitalización no es igual** en todos los municipios del top 10. Por sexo, edad y % hosp. suelen ser **parecidos** entre M y F; las gestantes aparecen solo en F.

## 8.4. Estacionalidad y brotes (H1)

**Hipótesis H1:** existe estacionalidad (concentración en ciertos meses/semanas) **en el Valle**.

**Pregunta guía:** ¿El patrón temporal se mantiene al desagregar por área (urbana/rural)?


In [ ]:
# Casos por mes y por área (urbana/rural) — Valle
area_map = {1: "1 — Urbana", 2: "2 — Rural", 3: "3 — Urbana-rural"}
mes_map = {
    1: "Ene", 2: "Feb", 3: "Mar", 4: "Abr", 5: "May", 6: "Jun",
    7: "Jul", 8: "Ago", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dic",
}

casos_mes_area = pd.crosstab(df_limpio["mes"], df_limpio["AREA"]).sort_index()
casos_mes_area = casos_mes_area.rename(columns=area_map)

ax = casos_mes_area.plot(
    kind="bar", stacked=True,
    title=f"Casos por mes según área — Valle {ANIO}", figsize=(10, 4),
)
ax.set_xlabel("Mes")
ax.set_ylabel("Casos")
ax.set_xticklabels([mes_map.get(m, str(m)) for m in casos_mes_area.index], rotation=0)
plt.legend(title="Área")
plt.tight_layout()
plt.show()

# Curva de brote: semana vs volumen
casos_por_sem = df_limpio.groupby("SEMANA").size().reset_index(name="Total_Casos")
plt.figure(figsize=(9, 5))
sns.regplot(
    data=casos_por_sem, x="SEMANA", y="Total_Casos",
    scatter_kws={"alpha": 0.6, "color": "seagreen"},
    line_kws={"color": "darkred"},
)
r_temp = casos_por_sem["SEMANA"].corr(casos_por_sem["Total_Casos"])
plt.title(f"Valle: semana epidemiológica vs casos (r = {r_temp:.2f})")
plt.xlabel("Semana epidemiológica")
plt.ylabel("Número total de casos")
plt.tight_layout()
plt.show()


> **Lectura rápida (H1):** si el pico de inicio de año se ve también por área y **urbana** concentra la mayor parte, H1 se sostiene a escala departamental. Un r negativo semana–casos indica descenso de la curva a lo largo de 2025 (brote temprano, no uniforme todo el año).

## 8.5. Análisis territorial (H2) — ranking municipal

**Hipótesis H2:** heterogeneidad municipal (pocos municipios concentran ≥ 60 % de los casos).

No construimos mapas con shapefiles. El entregable es el **ranking** y el share acumulado.


In [ ]:
# Ranking municipal completo (top 15) con hospitalización
ranking_mun = (
    df_limpio.groupby("Municipio_ocurrencia")
    .agg(
        casos=("CONSECUTIVE", "count"),
        pct_hospitalizado=("hospitalizado", "mean"),
        edad_mediana=("edad_anios", "median"),
        n_gestantes=("gestante", "sum"),
    )
    .sort_values("casos", ascending=False)
    .head(15)
)
ranking_mun["% Valle"] = (ranking_mun["casos"] / len(df_limpio) * 100).round(2)
ranking_mun["% acumulado"] = ranking_mun["casos"].cumsum() / len(df_limpio) * 100
ranking_mun["pct_hospitalizado"] = (ranking_mun["pct_hospitalizado"] * 100).round(1)
ranking_mun.round(2)


## 8.6. Perfil demográfico y poblaciones especiales (H3)

**Hipótesis H3:** edad, sexo, gestantes y hospitalización **difieren entre municipios-foco** y grupos de edad.

**Preguntas guía:**
- ¿La edad mediana cambia entre los municipios con más casos?
- ¿Qué grupo etario aporta más casos y más hospitalización?
- ¿Las gestantes tienen un % de hospitalización distinto al resto de mujeres?


In [ ]:
# H3 — edad por municipio-foco, grupos etarios y gestantes
top7 = df_limpio["Municipio_ocurrencia"].value_counts().head(7).index
df_top7 = df_limpio[df_limpio["Municipio_ocurrencia"].isin(top7)].copy()

plt.figure(figsize=(12, 5))
sns.boxplot(
    data=df_top7, x="Municipio_ocurrencia", y="edad_anios",
    order=top7, hue="Municipio_ocurrencia", palette="Set2", legend=False,
)
plt.xticks(rotation=30, ha="right")
plt.title("Edad en los 7 municipios con más casos (Valle 2025)")
plt.ylabel("Edad (años)")
plt.tight_layout()
plt.show()

# Grupos de edad: volumen vs hospitalización
resumen_edad = (
    df_limpio.groupby("grupo_edad", observed=False)
    .agg(
        casos=("CONSECUTIVE", "count"),
        pct_hospitalizado=("hospitalizado", "mean"),
        pct_confirmado=("confirmado", "mean"),
    )
)
resumen_edad["% Valle"] = resumen_edad["casos"] / len(df_limpio) * 100
display(resumen_edad.round(3))

resumen_edad["casos"].plot(kind="bar", color="crimson", title="Casos por grupo de edad — Valle 2025")
plt.ylabel("Casos")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

(resumen_edad["pct_hospitalizado"] * 100).plot(
    kind="bar", color="darkorange", title="% hospitalización por grupo de edad — Valle 2025"
)
plt.ylabel("%")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Gestantes vs otras mujeres
mujeres = df_limpio[df_limpio["SEXO"].astype(str).str.upper().isin(["F", "FEMENINO"])].copy()
cmp_gest = mujeres.groupby("gestante").agg(
    casos=("CONSECUTIVE", "count"),
    edad_media=("edad_anios", "mean"),
    pct_hospitalizado=("hospitalizado", "mean"),
    pct_confirmado=("confirmado", "mean"),
).round(3)
cmp_gest.index = cmp_gest.index.map({True: "Gestante", False: "Mujer no gestante"})
print("===== Mujeres: gestantes vs no gestantes =====")
display(cmp_gest)

# Otros grupos poblacionales SIVIGILA (1 = Sí)
gp_cols = [c for c in df_limpio.columns if c.startswith("GP_") and c != "GP_GESTAN"]
if gp_cols:
    resumen_gp = pd.Series({c: int((df_limpio[c] == 1).sum()) for c in gp_cols}).sort_values(ascending=False)
    print("\n===== Otros grupos poblacionales (conteo GP_*=1) =====")
    print(resumen_gp[resumen_gp > 0])


**Lectura H3 (Valle del Cauca, 2025):**

- Los boxplots de edad por municipio muestran si los focos tienen **poblaciones distintas** (p. ej. medianas más altas o más niños).
- `grupo_edad` responde a “ciertas poblaciones”: no basta el total departamental; niños, adultos y 60+ pueden tener **% hosp. distinto**.
- Gestantes son un subconjunto pequeño pero clínicamente prioritario; comparar su % hosp. con el resto de mujeres es el KPI de población especial.
- Los flags `GP_*` (desplazados, migrantes, indígenas, etc.) se reportan como conteo: suelen ser pocos registros, útiles para vigilancia diferencial, no para un modelo de volumen.

**Conclusión H3 (tras ejecutar):** si edad mediana y/o % hosp. cambian entre municipios-foco o entre grupos etarios/gestantes, el perfil **no es homogéneo** dentro del Valle → priorizar por municipio **y** por población.


**Interpretación bivariada (síntesis) — cierre de preguntas e hipótesis:**

| Hipótesis | Preguntas que cierra | Resultado | Evidencia en el notebook |
|-----------|----------------------|-----------|---------------------------|
| **H1** | Distribución por mes/semana; patrón por área | **A contrastar al ejecutar** | §§ 7.1 y 8.4: ventana de pico y r semana–casos |
| **H2** | Municipios con más casos; share top 10 ≥ 60 %; hosp. por municipio | **A contrastar al ejecutar** | §§ 7.3, 8.3 y 8.5: ranking y % acumulado |
| **H3** | Edad/sexo/grupos etarios/gestantes entre focos | **A contrastar al ejecutar** | § 8.6: boxplots, tabla etaria, gestantes vs no |

**Notas de lectura:**
- Correlación `SEMANA`–`mes` es alta **por construcción**; no es un hallazgo epidemiológico.
- Dispersión edad×semana: no separa bien hospitalizados a nivel individual → conviene tasas agregadas.
- Alcance: solo `Datos_2025_210.xlsx`, universo **Valle 2025** (ocurrencia).


---
# 9. Conclusiones

> *Etapa del flujo: **Conclusiones***

La pregunta SMART de **este taller** (alcance reducido) es:

> En el Valle del Cauca durante 2025, ¿un conjunto reducido de municipios (top 10) concentra al menos el 60 % de las notificaciones de dengue, con una ventana de pico identificable y un perfil demográfico/clínico (edad, sexo, hospitalización, gestantes) que permita priorizar focos municipales?

Eso **sí es medible** con el Excel. No afirmamos precisión de modelo, reducción de mortalidad ni de tiempo de respuesta (visión de proyecto del Taller 1).

## 9.1. Contraste de las hipótesis

| Hipótesis | Enunciado | Cómo se decide |
|-----------|-----------|----------------|
| **H1** | Estacionalidad en meses/semanas **en el Valle** | Confirmada si hay pico claro (p. ej. inicio de año) en §§ 7.1 y 8.4 |
| **H2** | Heterogeneidad municipal (share top 10 ≥ 60 %; hosp. desigual) | Confirmada si el % acumulado del top 10 supera el umbral SMART (§§ 6.3, 7.5) |
| **H3** | Perfil demográfico y poblaciones no homogéneo entre focos/grupos | Confirmada si edad o % hosp. cambian por municipio, grupo etario o gestantes (§ 8.6) |

> **Fuente:** solo `data/raw/Datos_2025_210.xlsx`, filtrado a ocurrencia en Valle y `FEC_NOT` 2025.

## 9.2. Sobre la calidad del conjunto de datos

- **Universo:** archivo nacional ~120.5k filas; tras `FEC_NOT` 2025 y filtro Valle quedan los casos departamentales (en Taller 1: ~7.4k / ~6.3 % del país — verificar el print de §6.4).
- **Completitud:** `FEC_NOT`, municipio, `SEXO`, `EDAD`, `PAC_HOS` bien poblados. `FEC_DEF` nulo. `sem_ges` nulo estructural fuera de gestantes.
- **Formato:** fechas parseadas; `edad_anios`, `mes`, `hospitalizado`, `confirmado`, `gestante`, `grupo_edad`.
- **Relevancia para un modelo futuro en Valle:** tiempo (`SEMANA`/`mes`), municipio y perfil (`edad_anios`, `SEXO`, hospitalización, gestante).

## 9.3. Limitaciones y recomendaciones

- Un solo año (2025), un departamento, una fuente (SIVIGILA).
- **`FEC_DEF` 100 % nulo** → no hay mortalidad.
- Residencia y ocurrencia no siempre coinciden; priorizamos **ocurrencia** porque define el foco.
- Siguiente paso: agregar a **municipio–semana en el Valle** y entrenar una **regresión de casos** sobre esa grilla.

## 9.4. Respuesta a la pregunta de negocio

Con el Excel 2025 acotado al Valle podemos **medir** concentración municipal, ventana temporal y diferencias de perfil (edad, sexo, hospitalización, gestantes). Eso apoya priorizar **municipios-foco** y **poblaciones** en la red departamental.

No afirmamos aún un modelo con precisión > 80 % ni reducción de mortalidad: eso sigue fuera de este EDA. El Taller 2 deja el recorte territorial y poblacional listo para esa fase.


---
# 10. Evidencia de uso de IA generativa y justificación

**Grupo 3 — integrantes**

| Integrante |
|------------|
| Juan Manuel Román Villa |
| Dora Valencia Martínez |
| Julian Aguilar Mayorga |
| Camilo Percy Ocampo |
| Viviana Fernández Payan |
| Giovanni Jaramillo Bolaños |
| Victor Manuel Hurtado López |

## 10.1. Por qué usamos ciencia de datos / IA (resumen)

El Taller 1 mostró que la carga **no es uniforme** en el país. El Taller 2 baja el alcance a **Valle 2025** para validar calidad, estacionalidad municipal, focos y poblaciones (edad, sexo, gestantes) sobre la misma fuente SIVIGILA. Después: regresión agregada municipio–semana **dentro del Valle**.

## 10.2. Evidencia de uso de IA generativa

| Pregunta | Respuesta |
|----------|-----------|
| **Herramienta usada** | Cursor (agente de código) |
| **Qué partes asistió la IA** | Estructura heredada del Taller 1 y de la guía; recorte SMART/KPI/hipótesis a Valle 2025; propuesta de arquitectura (PDF + §2); limpieza y features (`gestante`, `grupo_edad`); gráficos municipales y de poblaciones; borradores de interpretación |
| **Qué validamos nosotros** | Filtro de ocurrencia vs residencia; umbral SMART del 60 % municipal; diccionario (`GP_GESTAN`, `sem_ges`); re-ejecución de celdas clave y cifras impresas |
| **Iteraciones relevantes** | Bajar alcance de país → departamento; SMART medible sin modelo; H3 de “Valle vs nacional” a “poblaciones dentro del Valle”; no pisar el parquet nacional del Taller 1 |
